# ✈️ Flight Delay Prediction Pipeline

This notebook implements an end-to-end, production-ready machine learning pipeline to predict flight delays (>= 15 minutes). 

### Key Features:
- **Zero Data Leakage:** All preprocessing, transformations, and target encoding are isolated within Scikit-Learn `Pipeline`s.
- **Advanced Feature Engineering:** Calculates Haversine geospatial distances, time-based cyclic encodings, and airport traffic load.
- **Robust Cross-Validation:** Uses `StratifiedKFold` to handle severe class imbalances and safely computes `ROC-AUC`.
- **Threshold Optimization:** Optimizes prediction thresholds automatically using custom objective equations.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import logging
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, precision_recall_curve, roc_curve
)
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.calibration import CalibratedClassifierCV
import joblib
import warnings

warnings.filterwarnings('ignore')

# ── DIRECTORY SETUP ──────────────────────────────────────────────────────────
# Resolve project root robustly across VS Code, JupyterLab, and classic Jupyter
def resolve_project_root():
    # 1) VS Code sets __vsc_ipynb_file__ to the notebook's absolute path
    try:
        nb_path = Path(globals().get('__vsc_ipynb_file__') or '')
        if nb_path.exists():
            return nb_path.parent
    except Exception:
        pass
    # 2) Walk from cwd upward looking for project-specific markers
    markers = ('flight_delay_assignment.ipynb', 'flight_delay_assignment.py', 'flights.csv')
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if sum((candidate / m).exists() for m in markers) >= 2:
            return candidate
    # 3) Last resort: use cwd
    return Path.cwd()

PROJECT_ROOT = resolve_project_root()
OUTPUT_ROOT  = PROJECT_ROOT / 'output'
PLOTS_DIR    = OUTPUT_ROOT / 'plots'
MODELS_DIR   = OUTPUT_ROOT / 'models'
LOGS_DIR     = OUTPUT_ROOT / 'logs'
METRICS_DIR  = OUTPUT_ROOT / 'metrics'
REPORTS_DIR  = OUTPUT_ROOT / 'reports'

for _d in [PLOTS_DIR, MODELS_DIR, LOGS_DIR, METRICS_DIR, REPORTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ── LOGGING SETUP ────────────────────────────────────────────────────────────
logger = logging.getLogger('flight_delay_nb')
logger.setLevel(logging.INFO)
if logger.hasHandlers():
    logger.handlers.clear()

_fmt = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

# Stream handler (shows in Jupyter output)
_ch = logging.StreamHandler()
_ch.setLevel(logging.INFO)
_ch.setFormatter(_fmt)
logger.addHandler(_ch)

# File handler (writes to output/logs/pipeline.log)
_fh = logging.FileHandler(LOGS_DIR / 'pipeline.log', mode='w', encoding='utf-8')
_fh.setLevel(logging.INFO)
_fh.setFormatter(_fmt)
logger.addHandler(_fh)

logger.info(f'Project root   : {PROJECT_ROOT}')
logger.info(f'Artifacts saved: {OUTPUT_ROOT}')
print(f'Project root   : {PROJECT_ROOT}')
print(f'Artifacts saved: {OUTPUT_ROOT}')


## 1. Custom Scikit-Learn Transformers
We define robust classes to handle cyclic time features, geospatial distances, traffic load, and Out-Of-Fold (OOF) target encodings safely inside our pipeline.


In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

class FlightFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.origin_traffic_ = {}
        self.dest_traffic_ = {}
        self.route_freq_ = {}
        self.major_cities = ['New York', 'Los Angeles', 'Chicago', 'Atlanta', 'Dallas-Fort Worth', 'Dallas', 'Houston', 'Denver']
        
    def fit(self, X, y=None):
        self.origin_traffic_ = X['ORIGIN_AIRPORT'].value_counts().to_dict()
        self.dest_traffic_ = X['DESTINATION_AIRPORT'].value_counts().to_dict()
        
        route_str = X['ORIGIN_AIRPORT'].astype(str) + '_' + X['DESTINATION_AIRPORT'].astype(str)
        self.route_freq_ = route_str.value_counts().to_dict()
        return self

    def transform(self, X, y=None):
        X_out = X.copy()
        try:
            dep_str = X_out['SCHEDULED_DEPARTURE'].astype(float).astype(int).astype(str).str.zfill(4)
            X_out['HOUR'] = dep_str.str[:2].astype(float)
        except Exception:
            X_out['HOUR'] = 0.0
        X_out['HOUR'] = X_out['HOUR'].fillna(0)
        
        # Cyclic encoding
        X_out['MONTH'] = X_out['MONTH'].fillna(1)
        X_out['DAY'] = X_out['DAY'].fillna(1)
        X_out['DAY_OF_WEEK'] = X_out['DAY_OF_WEEK'].fillna(1)

        X_out['MONTH_sin'] = np.sin(2 * np.pi * X_out['MONTH'] / 12.0)
        X_out['MONTH_cos'] = np.cos(2 * np.pi * X_out['MONTH'] / 12.0)
        X_out['DAY_sin'] = np.sin(2 * np.pi * X_out['DAY'] / 31.0)
        X_out['DAY_cos'] = np.cos(2 * np.pi * X_out['DAY'] / 31.0)
        X_out['DAY_OF_WEEK_sin'] = np.sin(2 * np.pi * X_out['DAY_OF_WEEK'] / 7.0)
        X_out['DAY_OF_WEEK_cos'] = np.cos(2 * np.pi * X_out['DAY_OF_WEEK'] / 7.0)
        X_out['HOUR_sin'] = np.sin(2 * np.pi * X_out['HOUR'] / 24.0)
        X_out['HOUR_cos'] = np.cos(2 * np.pi * X_out['HOUR'] / 24.0)
        
        # Flags
        X_out['is_weekend'] = (X_out['DAY_OF_WEEK'] > 5).astype(int)
        X_out['is_peak_hour'] = (((X_out['HOUR'] >= 7) & (X_out['HOUR'] <= 9)) | ((X_out['HOUR'] >= 16) & (X_out['HOUR'] <= 19))).astype(int)
        X_out['is_night_flight'] = ((X_out['HOUR'] >= 22) | (X_out['HOUR'] <= 5)).astype(int)
        
        # Speed
        time_valid = X_out['SCHEDULED_TIME'].replace(0, np.nan)
        X_out['SPEED'] = X_out['DISTANCE'] / time_valid
        
        # Advanced Features
        X_out['ROUTE'] = X_out['ORIGIN_AIRPORT'].astype(str) + '_' + X_out['DESTINATION_AIRPORT'].astype(str)
        X_out['GEO_DISTANCE'] = haversine(
            X_out['ORIGIN_LATITUDE'].fillna(0), X_out['ORIGIN_LONGITUDE'].fillna(0), 
            X_out['DEST_LATITUDE'].fillna(0), X_out['DEST_LONGITUDE'].fillna(0)
        )
        
        X_out['SAME_STATE'] = (X_out['ORIGIN_STATE'] == X_out['DEST_STATE']).astype(int)
        X_out['MAJOR_CITY'] = (X_out['ORIGIN_CITY'].isin(self.major_cities) | X_out['DEST_CITY'].isin(self.major_cities)).astype(int)
        
        orig_t = X_out['ORIGIN_AIRPORT'].map(self.origin_traffic_).fillna(1)
        dest_t = X_out['DESTINATION_AIRPORT'].map(self.dest_traffic_).fillna(1)
        X_out['AIRPORT_TRAFFIC'] = orig_t + dest_t
        X_out['AIRPORT_IMPORTANCE'] = (orig_t * dest_t) / 1000.0  
        X_out['ROUTE_FREQUENCY'] = X_out['ROUTE'].map(self.route_freq_).fillna(1)
        X_out['DISTANCE_CATEGORY'] = pd.cut(X_out['DISTANCE'], bins=[0, 500, 1500, 10000], labels=[1, 2, 3]).astype(float).fillna(2)
        X_out['LONG_HAUL'] = (X_out['GEO_DISTANCE'] > 2000).astype(int)
        
        # Interaction features
        X_out['TIME_DISTANCE'] = X_out['HOUR'] * X_out['DISTANCE']
        X_out['TRAFFIC_PRESSURE'] = X_out['AIRPORT_TRAFFIC'] * X_out['is_peak_hour']
        X_out['ROUTE_COMPLEXITY'] = X_out['ROUTE_FREQUENCY'] / (X_out['DISTANCE'] + 1)
        
        cols_to_drop = ['MONTH', 'DAY', 'DAY_OF_WEEK', 'HOUR', 'SCHEDULED_DEPARTURE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_ARRIVAL', 'ORIGIN_LATITUDE', 'ORIGIN_LONGITUDE', 'DEST_LATITUDE', 'DEST_LONGITUDE', 'ORIGIN_CITY', 'ORIGIN_STATE', 'DEST_CITY', 'DEST_STATE', 'AIRLINE_NAME', 'AIRLINE']
        return X_out.drop(columns=[c for c in cols_to_drop if c in X_out.columns])

class OOFTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols, cv=5, alpha=10):
        self.cols = cols
        self.cv = cv
        self.alpha = alpha

    def fit(self, X, y):
        y_series = pd.Series(y, index=X.index)
        self.global_means_ = {}
        self.category_means_ = {}
        
        for col in self.cols:
            self.global_means_[col] = y_series.mean()
            stats = y_series.groupby(X[col]).agg(['mean', 'count'])
            self.category_means_[col] = ((stats['count'] * stats['mean'] + self.alpha * self.global_means_[col]) / (stats['count'] + self.alpha)).to_dict()
        return self

    def transform(self, X, y=None):
        X_out = X.copy()
        if y is not None:
            y_series = pd.Series(y, index=X.index)
            kf = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=42)
            for col in self.cols:
                oof_encoded = pd.Series(np.nan, index=X.index)
                for train_idx, val_idx in kf.split(X, y):
                    train_x, val_x = X.iloc[train_idx], X.iloc[val_idx]
                    train_y = y_series.iloc[train_idx]
                    stats = train_y.groupby(train_x[col]).agg(['mean', 'count'])
                    smooth_mean = (stats['count'] * stats['mean'] + self.alpha * self.global_means_[col]) / (stats['count'] + self.alpha)
                    oof_encoded.iloc[val_idx] = val_x[col].map(smooth_mean).fillna(self.global_means_[col])
                X_out[col + '_TE'] = oof_encoded
                X_out.drop(columns=[col], inplace=True)
        else:
            for col in self.cols:
                X_out[col + '_TE'] = X[col].map(self.category_means_[col]).fillna(self.global_means_[col])
                X_out.drop(columns=[col], inplace=True)
        return X_out

class FullPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.engineer = FlightFeatureEngineer()
        self.imputer_num = SimpleImputer(strategy='median')
        self.imputer_cat = SimpleImputer(strategy='constant', fill_value='Unknown')
        self.te = OOFTargetEncoder(cols=['AIRLINE', 'ROUTE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'], alpha=10)
        self.scaler = StandardScaler()
        
    def fit(self, X, y):
        X_temp = X.copy()
        X_temp['ROUTE'] = X_temp['ORIGIN_AIRPORT'].astype(str) + '_' + X_temp['DESTINATION_AIRPORT'].astype(str)
        cat_cols = ['AIRLINE', 'ROUTE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']
        self.imputer_cat.fit(X_temp[cat_cols])
        
        cat_imputed = pd.DataFrame(self.imputer_cat.transform(X_temp[cat_cols]), columns=cat_cols, index=X.index)
        self.te.fit(cat_imputed, y)
        
        X_eng = self.engineer.fit_transform(X)
        num_cols = [c for c in X_eng.columns if c not in cat_cols and c != 'ROUTE']
        self.imputer_num.fit(X_eng[num_cols])
        
        X_combined = pd.concat([pd.DataFrame(self.imputer_num.transform(X_eng[num_cols]), columns=num_cols, index=X.index), self.te.transform(cat_imputed, y)], axis=1)
        self.scaler.fit(X_combined)
        self.final_cols = X_combined.columns
        return self
        
    def transform(self, X, y=None):
        X_temp = X.copy()
        X_temp['ROUTE'] = X_temp['ORIGIN_AIRPORT'].astype(str) + '_' + X_temp['DESTINATION_AIRPORT'].astype(str)
        cat_cols = ['AIRLINE', 'ROUTE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']
        cat_imputed = pd.DataFrame(self.imputer_cat.transform(X_temp[cat_cols]), columns=cat_cols, index=X.index)
        
        X_eng = self.engineer.transform(X)
        num_cols = [c for c in X_eng.columns if c not in cat_cols and c != 'ROUTE']
        
        X_combined = pd.concat([pd.DataFrame(self.imputer_num.transform(X_eng[num_cols]), columns=num_cols, index=X.index), self.te.transform(cat_imputed, y)], axis=1)
        return pd.DataFrame(self.scaler.transform(X_combined), columns=X_combined.columns, index=X.index)

class WeightedAdaBoost(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator=None, n_estimators=400, learning_rate=0.03, random_state=42):
        if estimator is None:
            estimator = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20)
        self.estimator = estimator
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.model = AdaBoostClassifier(estimator=self.estimator, n_estimators=self.n_estimators, learning_rate=self.learning_rate, random_state=self.random_state)

    def fit(self, X, y):
        self.model.fit(X, y, sample_weight=np.where(y == 1, 2.0, 1.0))
        self.classes_ = self.model.classes_
        return self

    def predict(self, X): return self.model.predict(X)
    def predict_proba(self, X): return self.model.predict_proba(X)
    @property
    def feature_importances_(self): return self.model.feature_importances_


## 2. Safe Data Loading & Merging
We load flight data and safely attach `airports` and `airlines` context mappings without causing target leakage.


In [ ]:
# Load Datasets
flights = pd.read_csv('flights.csv', usecols=[
    'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'ORIGIN_AIRPORT', 
    'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'SCHEDULED_ARRIVAL', 
    'SCHEDULED_TIME', 'DISTANCE', 'ARRIVAL_DELAY'
])
airlines = pd.read_csv('airlines.csv')
airports = pd.read_csv('airports.csv')

flights = flights.dropna(subset=['ARRIVAL_DELAY'])
flights['DELAYED'] = (flights['ARRIVAL_DELAY'] >= 15).astype(int)

# Merge Airlines
flights = flights.merge(airlines, left_on='AIRLINE', right_on='IATA_CODE', how='left')
flights.rename(columns={'AIRLINE_y': 'AIRLINE_NAME', 'AIRLINE_x': 'AIRLINE'}, inplace=True)
if 'IATA_CODE' in flights.columns: flights.drop(columns=['IATA_CODE'], inplace=True)

# Merge Origin Airports
flights = flights.merge(airports[['IATA_CODE', 'CITY', 'STATE', 'LATITUDE', 'LONGITUDE']], 
                        left_on='ORIGIN_AIRPORT', right_on='IATA_CODE', how='left')
flights.rename(columns={'CITY': 'ORIGIN_CITY', 'STATE': 'ORIGIN_STATE', 'LATITUDE': 'ORIGIN_LATITUDE', 'LONGITUDE': 'ORIGIN_LONGITUDE'}, inplace=True)
if 'IATA_CODE' in flights.columns: flights.drop(columns=['IATA_CODE'], inplace=True)

# Merge Dest Airports
flights = flights.merge(airports[['IATA_CODE', 'CITY', 'STATE', 'LATITUDE', 'LONGITUDE']], 
                        left_on='DESTINATION_AIRPORT', right_on='IATA_CODE', how='left')
flights.rename(columns={'CITY': 'DEST_CITY', 'STATE': 'DEST_STATE', 'LATITUDE': 'DEST_LATITUDE', 'LONGITUDE': 'DEST_LONGITUDE'}, inplace=True)
if 'IATA_CODE' in flights.columns: flights.drop(columns=['IATA_CODE'], inplace=True)

X_full = flights.drop(columns=['ARRIVAL_DELAY', 'DELAYED'])
y_full = flights['DELAYED']


## 2b. Exploratory Data Analysis


In [ ]:
# EDA: Exploratory Data Analysis
logger.info('EDA: Generating exploratory visualizations')

eda = flights.copy()
try:
    eda['HOUR'] = eda['SCHEDULED_DEPARTURE'].astype(float).astype(int).astype(str).str.zfill(4).str[:2].astype(int)
except Exception:
    eda['HOUR'] = 0

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold')

# 1a. Target class distribution
ax = axes[0, 0]
counts = eda['DELAYED'].value_counts().sort_index()
labels = ['On-Time', 'Delayed']
bar_colors = ['#2196F3', '#F44336']
bars = ax.bar(labels, counts.values, color=bar_colors, edgecolor='white', width=0.5)
for bar, cnt in zip(bars, counts.values):
    pct = 100 * cnt / len(eda)
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + len(eda)*0.005,
            f'{cnt:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Target Class Distribution', fontweight='bold')
ax.set_ylabel('Number of Flights')
ax.set_ylim(0, counts.max() * 1.18)
ax.spines[['top','right']].set_visible(False)

# 1b. Delay rate by hour
ax = axes[0, 1]
hourly = eda.groupby('HOUR')['DELAYED'].mean() * 100
ax.plot(hourly.index, hourly.values, marker='o', color='#FF9800', linewidth=2, markersize=5)
ax.fill_between(hourly.index, hourly.values, alpha=0.15, color='#FF9800')
ax.axhline(hourly.mean(), color='red', linestyle='--', linewidth=1, label=f'Mean: {hourly.mean():.1f}%')
ax.set_title('Delay Rate by Hour of Day', fontweight='bold')
ax.set_xlabel('Hour (24h)')
ax.set_ylabel('Delay Rate (%)')
ax.legend()
ax.spines[['top','right']].set_visible(False)

# 1c. Delay rate by month
ax = axes[1, 0]
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = eda.groupby('MONTH')['DELAYED'].mean() * 100
mcols = ['#F44336' if v == monthly.max() else '#5C6BC0' for v in monthly.values]
ax.bar([month_names[m-1] for m in monthly.index], monthly.values, color=mcols, edgecolor='white')
ax.axhline(monthly.mean(), color='gray', linestyle='--', linewidth=1, label=f'Mean: {monthly.mean():.1f}%')
ax.set_title('Delay Rate by Month', fontweight='bold')
ax.set_ylabel('Delay Rate (%)')
ax.legend()
ax.spines[['top','right']].set_visible(False)

# 1d. Delay rate by airline (top 10)
ax = axes[1, 1]
acol = 'AIRLINE_NAME' if 'AIRLINE_NAME' in eda.columns else 'AIRLINE'
airline_delay = (eda.groupby(acol)['DELAYED'].mean() * 100).sort_values(ascending=False).head(10)
acols = ['#F44336' if v == airline_delay.max() else '#26A69A' for v in airline_delay.values]
ax.barh(airline_delay.index[::-1], airline_delay.values[::-1], color=acols[::-1], edgecolor='white')
ax.axvline(airline_delay.mean(), color='gray', linestyle='--', linewidth=1)
ax.set_title('Delay Rate by Airline (Top 10)', fontweight='bold')
ax.set_xlabel('Delay Rate (%)')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'eda_overview.png', dpi=150, bbox_inches='tight')
logger.info(f"Saved EDA -> {PLOTS_DIR / 'eda_overview.png'}")
plt.show()
print(f'[OK] EDA saved -> {PLOTS_DIR}')


## 3. Subsampling & Preprocessing
To manage memory and compute times, we subsample the data and process it strictly separated from the test-set.


In [ ]:
SAMPLE_SIZE = 50000 
if len(flights) > SAMPLE_SIZE:
    _, X_full, _, y_full = train_test_split(X_full, y_full, test_size=SAMPLE_SIZE, stratify=y_full, random_state=42)

# Global Test Split (never touched until final validation)
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, stratify=y_full, random_state=42)
X_test_original = X_test.copy()

# Preprocess
preprocessor = FullPreprocessor()
X_train_preprocessed = preprocessor.fit_transform(X_train, y_train)
X_train_preprocessed.head()


## 4. Forward Feature Selection
Extracting the top 22 most highly correlated engineered features via logistic sequential testing.


In [ ]:
lr_evaluator = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
sfs = SequentialFeatureSelector(lr_evaluator, n_features_to_select=22, direction='forward', cv=5, n_jobs=-1)

sfs.fit(X_train_preprocessed, y_train)
selected_features = X_train_preprocessed.columns[sfs.get_support()].tolist()

print(f"Selected Features ({len(selected_features)}): {selected_features}")
X_train_sel = X_train_preprocessed[selected_features]


## 5. Model Training & Cross-Validation
Evaluating models securely. A strict manual `StratifiedKFold(n_splits=3)` ensures `NaN` values from highly-imbalanced folds do not disrupt the pipeline evaluation logic.


In [ ]:
models = {
    'Logistic Regression': Pipeline([('model', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))]),
    'Polynomial Regression': Pipeline([
        ('poly', PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)),
        ('model', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
    ]),
    'AdaBoost': Pipeline([
        ('model', WeightedAdaBoost(estimator=DecisionTreeClassifier(max_depth=4, min_samples_leaf=20), n_estimators=400, learning_rate=0.03))
    ])
}

model_results = {}
best_f1 = 0
best_model_name = ""

kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for name, pipeline in models.items():
    print(f"--- Cross-Validating {name} ---")
    fold_f1s = []
    fold_aucs = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_sel, y_train)):
        X_fold_train, X_fold_val = X_train_sel.iloc[train_idx], X_train_sel.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        unique_classes, counts = np.unique(y_fold_val, return_counts=True)
        if len(unique_classes) < 2:
            print(f"[{name}] Fold {fold+1} contains only one class -> AUC undefined!")
            
        pipeline.fit(X_fold_train, y_fold_train)
        try:
            y_pred = pipeline.predict(X_fold_val)
            y_proba = pipeline.predict_proba(X_fold_val)[:, 1]
            
            auc = np.nan if len(unique_classes) < 2 else roc_auc_score(y_fold_val, y_proba)
            f1 = f1_score(y_fold_val, y_pred)
            
            fold_f1s.append(f1)
            fold_aucs.append(auc)
        except Exception as e:
            print(f"[{name}] Fold {fold+1} prediction failed: {str(e)}")
            raise e 
            
    mean_f1 = np.nanmean(fold_f1s)
    mean_auc = np.nanmean(fold_aucs)
    
    model_results[name] = {'F1 (CV)': mean_f1, 'ROC-AUC (CV)': mean_auc}
    print(f"{name} -> Mean CV F1: {mean_f1:.4f}, Mean CV AUC: {mean_auc:.4f}")
    
    pipeline.fit(X_train_sel, y_train)
    if mean_f1 > best_f1:
        best_f1 = mean_f1
        best_model_name = name

# Display Summary
df_results = pd.DataFrame(model_results).T
display(df_results)


## 5b. Model Comparison & Per-Model Curves


In [ ]:
# Per-model OOF ROC, PR curves and comparison bar chart
from sklearn.metrics import precision_score as _ps, accuracy_score as _as
logger.info('Generating model comparison visualizations')

model_full_metrics = {}
fig_roc, ax_roc = plt.subplots(figsize=(7, 5))
fig_pr,  ax_pr  = plt.subplots(figsize=(7, 5))
pal3 = ['#2196F3', '#FF9800', '#F44336']

for (name, pipeline), color in zip(models.items(), pal3):
    oof_probs = cross_val_predict(pipeline, X_train_sel, y_train,
                                  cv=3, method='predict_proba', n_jobs=-1)[:, 1]
    oof_preds = (oof_probs >= 0.5).astype(int)

    fpr, tpr, _ = roc_curve(y_train, oof_probs)
    auc_val     = roc_auc_score(y_train, oof_probs)
    ax_roc.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=color, linewidth=2)

    prec_arr, rec_arr, _ = precision_recall_curve(y_train, oof_probs)
    pr_auc = float(np.trapz(prec_arr[::-1], rec_arr[::-1]))
    ax_pr.plot(rec_arr, prec_arr, label=f'{name} (PR-AUC={pr_auc:.3f})', color=color, linewidth=2)

    model_full_metrics[name] = {
        'Accuracy' : _as(y_train, oof_preds),
        'Precision': _ps(y_train, oof_preds, zero_division=0),
        'Recall'   : recall_score(y_train, oof_preds),
        'F1'       : f1_score(y_train, oof_preds),
        'ROC-AUC'  : auc_val,
    }

# ROC plot
ax_roc.plot([0,1],[0,1],'--', color='gray', linewidth=1)
ax_roc.set_xlabel('False Positive Rate')
ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC Curves - All Models (3-Fold OOF)', fontweight='bold')
ax_roc.legend()
ax_roc.spines[['top','right']].set_visible(False)
fig_roc.tight_layout()
fig_roc.savefig(PLOTS_DIR / 'roc_curves_all_models.png', dpi=150, bbox_inches='tight')
logger.info(f"Saved -> {PLOTS_DIR / 'roc_curves_all_models.png'}")
plt.show()

# PR plot
baseline = float(y_train.mean())
ax_pr.axhline(baseline, color='gray', linestyle='--', linewidth=1, label=f'Baseline ({baseline:.2f})')
ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.set_title('Precision-Recall Curves - All Models (3-Fold OOF)', fontweight='bold')
ax_pr.legend()
ax_pr.spines[['top','right']].set_visible(False)
fig_pr.tight_layout()
fig_pr.savefig(PLOTS_DIR / 'pr_curves_all_models.png', dpi=150, bbox_inches='tight')
logger.info(f"Saved -> {PLOTS_DIR / 'pr_curves_all_models.png'}")
plt.show()

# Model comparison bar chart
df_comp = pd.DataFrame(model_full_metrics).T
df_comp.to_csv(METRICS_DIR / 'model_comparison.csv')

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df_comp))
width = 0.15
metric_colors = ['#2196F3','#FF9800','#4CAF50','#F44336','#9C27B0']
for i, (metric, mc) in enumerate(zip(df_comp.columns, metric_colors)):
    offset = (i - 2) * width
    bars = ax.bar(x + offset, df_comp[metric], width, label=metric, color=mc, alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.2f}', ha='center', va='bottom', fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels(df_comp.index, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Model Comparison - All Metrics (3-Fold OOF)', fontweight='bold')
ax.legend(loc='upper right')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
logger.info(f"Saved -> {PLOTS_DIR / 'model_comparison.png'}")
plt.show()

display(df_comp.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=0))
print(f'[OK] Model comparison plots saved -> {PLOTS_DIR}')


## 6. Threshold Optimization & Probability Calibration
Fine-tuning probabilities into classifications based strictly on maximizing our objective equation: `Score = (0.7 * F1) + (0.5 * Recall)`.


In [ ]:
best_model_base = models[best_model_name]
calibrated_model = CalibratedClassifierCV(best_model_base, method='sigmoid', cv=3)
calibrated_model.fit(X_train_sel, y_train)

y_probs_cv = cross_val_predict(best_model_base, X_train_sel, y_train, cv=3, method='predict_proba', n_jobs=-1)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_train, y_probs_cv)

f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
optim_scores = (0.7 * f1_scores) + (0.5 * recalls[:-1])

valid_idx = np.where((thresholds >= 0.1) & (thresholds <= 0.9))[0]
if len(valid_idx) > 0:
    best_idx = valid_idx[np.argmax(optim_scores[valid_idx])]
    best_threshold = thresholds[best_idx]
    print(f"Optimal Threshold: {best_threshold:.4f} (Objective Score: {optim_scores[best_idx]:.4f})")
else:
    best_threshold = 0.5


## 7. Final Test Evaluation & Outputs
Validating the optimized threshold rules against the completely isolated test split and visualizing the results.


In [ ]:
X_test_preprocessed = preprocessor.transform(X_test, y=None)
X_test_sel = X_test_preprocessed[selected_features]

test_probs = calibrated_model.predict_proba(X_test_sel)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

test_metrics = {
    'accuracy': accuracy_score(y_test, test_preds),
    'precision': precision_score(y_test, test_preds),
    'f1': f1_score(y_test, test_preds),
    'recall': recall_score(y_test, test_preds),
    'auc': roc_auc_score(y_test, test_probs)
}

for k, v in test_metrics.items():
    print(f'Test {k.upper()}: {v:.4f}')
    logger.info(f'Test {k.upper()}: {v:.4f}')

# Save metrics CSV
pd.Series(test_metrics, name='value').to_csv(METRICS_DIR / 'test_metrics.csv', header=True)
logger.info(f"Saved metrics → {METRICS_DIR / 'test_metrics.csv'}")

# ── Confusion Matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_test, test_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('Final Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'final_confusion_matrix.png', dpi=150, bbox_inches='tight')
logger.info(f"Saved plot → {PLOTS_DIR / 'final_confusion_matrix.png'}")
plt.show()

# ── ROC Curve ────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, test_probs)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f"AUC = {test_metrics['auc']:.4f}")
ax.plot([0, 1], [0, 1], linestyle='--', color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'roc_curve.png', dpi=150, bbox_inches='tight')
logger.info(f"Saved plot → {PLOTS_DIR / 'roc_curve.png'}")
plt.show()

# ── Precision-Recall Curve ────────────────────────────────────────────────
precisions, recalls, _ = precision_recall_curve(y_test, test_probs)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recalls, precisions)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'precision_recall_curve.png', dpi=150, bbox_inches='tight')
logger.info(f"Saved plot → {PLOTS_DIR / 'precision_recall_curve.png'}")
plt.show()

print(f"\n✅ Metrics  → {METRICS_DIR}")
print(f"✅ Plots    → {PLOTS_DIR}")
print(f"✅ Log file → {LOGS_DIR / 'pipeline.log'}")


## 8. Exporting Serialized Pipelines
We group all artifacts, clusters, and the calibrated model inside a `joblib` object for future inference via the Streamlit web app.


In [ ]:
kmeans = KMeans(n_clusters=6, random_state=42)
train_clusters = kmeans.fit_predict(X_train_sel)

# ── Feature Importance (AdaBoost) ─────────────────────────────────────────
if best_model_name == 'AdaBoost':
    try:
        importances = best_model_base.named_steps['model'].feature_importances_
        idx = np.argsort(importances)[::-1][:20]
        top_feats = np.array(selected_features)[idx]
        top_imps  = importances[idx]

        fig, ax = plt.subplots(figsize=(10, 7))
        colors_fi = plt.cm.viridis(np.linspace(0.2, 0.9, len(top_feats)))
        ax.barh(top_feats[::-1], top_imps[::-1], color=colors_fi[::-1], edgecolor='white')
        ax.set_title('Top 20 Feature Importances (AdaBoost)', fontweight='bold', fontsize=13)
        ax.set_xlabel('Importance Score')
        ax.spines[['top','right']].set_visible(False)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
        logger.info(f"Saved -> {PLOTS_DIR / 'feature_importance.png'}")
        plt.show()
    except Exception as _e:
        logger.warning(f'Feature importance failed: {_e}')

# ── Cluster Distribution bar ────────────────────────────────────────────────
cluster_counts_arr = np.bincount(train_clusters)
fig, ax = plt.subplots(figsize=(8, 5))
cpal = plt.cm.Set2(np.linspace(0, 1, len(cluster_counts_arr)))
ax.bar(range(len(cluster_counts_arr)), cluster_counts_arr, color=cpal, edgecolor='white')
ax.set_title('Cluster Distribution', fontweight='bold')
ax.set_xlabel('Cluster ID')
ax.set_ylabel('Count')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'cluster_distribution.png', dpi=150, bbox_inches='tight')
logger.info(f"Saved -> {PLOTS_DIR / 'cluster_distribution.png'}")
plt.show()

# ── Cluster Delay Rate ──────────────────────────────────────────────────────
X_train_clustered = X_train_sel.copy()
X_train_clustered['CLUSTER'] = train_clusters
X_train_clustered['DELAYED'] = y_train.values
cluster_analysis = X_train_clustered.groupby('CLUSTER').mean()
cluster_analysis.to_csv(REPORTS_DIR / 'cluster_summary.csv')
logger.info(f"Saved report -> {REPORTS_DIR / 'cluster_summary.csv'}")

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(cluster_analysis.index, cluster_analysis['DELAYED'] * 100,
              color='salmon', edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}%", ha='center', va='bottom', fontsize=9)
ax.set_title('Delay Rate per Cluster', fontweight='bold')
ax.set_xlabel('Cluster ID')
ax.set_ylabel('Delay Rate (%)')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'cluster_delay_rate.png', dpi=150, bbox_inches='tight')
logger.info(f"Saved -> {PLOTS_DIR / 'cluster_delay_rate.png'}")
plt.show()

# ── Cluster PCA Scatter ─────────────────────────────────────────────────────
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train_sel.values)

fig, ax = plt.subplots(figsize=(9, 7))
scatter_pal = plt.cm.tab10(np.linspace(0, 0.6, 6))
for cid in range(6):
    mask = train_clusters == cid
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=8, alpha=0.4,
               color=scatter_pal[cid], label=f'Cluster {cid}')
ax.set_title('Cluster Visualization (PCA 2D Projection)', fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(markerscale=2, loc='best')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'cluster_pca_scatter.png', dpi=150, bbox_inches='tight')
logger.info(f"Saved -> {PLOTS_DIR / 'cluster_pca_scatter.png'}")
plt.show()

# ── Save final pipeline ─────────────────────────────────────────────────────
final_pipeline_obj = {
    'preprocessor'    : preprocessor,
    'feature_selector': sfs,
    'selected_features': selected_features,
    'base_model'      : best_model_base,
    'calibrated_model': calibrated_model,
    'best_threshold'  : best_threshold,
    'cluster_model'   : kmeans
}

model_path = MODELS_DIR / 'model.joblib'
joblib.dump(final_pipeline_obj, model_path)
logger.info(f'Pipeline saved -> {model_path}')
print(f'[OK] Pipeline saved   -> {model_path}')
print(f'[OK] Log file         -> {LOGS_DIR / "pipeline.log"}')
print(f'[OK] Reports          -> {REPORTS_DIR}')
print(f'[OK] Plots generated  -> {PLOTS_DIR}')
